In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp,col

spark = SparkSession.builder.getOrCreate()
tables = ["customers","sales","sales_orders"]
source_path = "/Volumes/ecommerce_analytics/bronze/raw_data/"
catalog  = "ecommerce_analytics"
schema = "bronze"

def add_meta_data_column(df):
    df = df.withColumn("last_update_ts" , current_timestamp())\
        .withColumn("file_path", col("_metadata.file_path"))
    return df

for table in tables:
    input_path = f"{source_path}{table}"
    df = spark.read.csv(input_path,inferSchema=True,header =True)

    df = add_meta_data_column(df)
    
    # # Drop existing table to allow schema change
    # spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.{table}")
    
    df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{schema}.{table}")    

In [0]:
df_customers = spark.read.table("ecommerce_analytics.bronze.customers")
df_sales = spark.read.table("ecommerce_analytics.bronze.sales")
df_sales_orders = spark.read.table("ecommerce_analytics.bronze.sales_orders")

In [0]:
df_customers.display()
df_sales.display()
df_sales_orders.display()